# DCC Mission 2 — 사전 생성 log-Mel 특징으로 학습
원본 36GB WAV 대신 Mac에서 미리 생성한 약 3.1GB `float16` 특징을 사용합니다. 학습 시에는 `float32`로 복원하며 SpecAugment와 Mixup은 매 epoch 새로 적용됩니다.

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

drive.mount('/content/drive')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU 없음')

## 경로 설정 및 Colab SSD로 복사
Google Drive의 `내 드라이브/DCC/` 아래에 `mission02.zip`과 `mission02_features/`가 있어야 합니다. 특징을 Drive에서 직접 읽으면 느리므로 `/content`로 먼저 복사합니다.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/DCC')
CODE_ZIP = DRIVE_ROOT / 'mission02.zip'
FEATURE_DRIVE_ROOT = DRIVE_ROOT / 'mission02_features'
CODE_ROOT = Path('/content/mission02')
FEATURE_ROOT = Path('/content/mission02_features')
OUTPUT_ROOT = DRIVE_ROOT / 'mission02_training_output'

assert CODE_ZIP.is_file(), CODE_ZIP
assert (FEATURE_DRIVE_ROOT / 'metadata.json').is_file(), FEATURE_DRIVE_ROOT
!unzip -q -o "{CODE_ZIP}" -d /content
!mkdir -p "{FEATURE_ROOT}"
!cp -R "{FEATURE_DRIVE_ROOT}/." "{FEATURE_ROOT}/"
!pip -q install -r "{CODE_ROOT}/requirements.txt"

In [ ]:
!python "{CODE_ROOT}/verify_features.py" "{FEATURE_ROOT}"

## 학습
Validation은 평가·early stopping·threshold 조정에만 사용되고 역전파에는 들어가지 않습니다. T4 메모리가 부족하면 batch size를 256으로 낮추세요.

In [ ]:
!python "{CODE_ROOT}/train.py" --feature-root "{FEATURE_ROOT}" --output-dir "{OUTPUT_ROOT}" --epochs 12 --batch-size 512 --num-workers 2 --resume

## Validation Accuracy 확인

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

history = pd.DataFrame(json.loads((OUTPUT_ROOT / 'history.json').read_text()))
display(history[['epoch', 'train_loss', 'valid_loss', 'valid_accuracy_0.5', 'valid_accuracy_tuned', 'valid_threshold']])
history.plot(x='epoch', y=['valid_accuracy_0.5', 'valid_accuracy_tuned'], marker='o', grid=True)
plt.show()

## 결과 다운로드
`best_model.pt`, `history.json`, `last_checkpoint.pt`는 매 epoch Drive에 직접 저장됩니다. 런타임이 끊기면 노트북을 다시 위에서부터 실행하세요. `--resume`이 마지막 완료 epoch 다음부터 이어서 학습합니다.

In [ ]:
from google.colab import files

files.download(str(OUTPUT_ROOT / 'best_model.pt'))
files.download(str(OUTPUT_ROOT / 'history.json'))